In [1]:
%load_ext autoreload
%autoreload 2

from fibsem import utils, acquire
from fibsem.milling import get_milling_stages, mill_stages
from fibsem.milling.strategy import register_strategy
from fibsem.structures import BeamType
from autolamella.protocol.validation import validate_protocol
from pprint import pprint

from adaptive_polish import strategy


Default configuration default-configuration. Configuration Path: c:\Users\dmv31621\AppData\Local\miniforge3\envs\adaptivepolish\lib\site-packages\fibsem\config\microscope-configuration.yaml


c:\Users\dmv31621\AppData\Local\miniforge3\envs\adaptivepolish\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# connect to microscope
microscope, settings = utils.setup_session(
    config_path=r"C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\fibsem\fibsem\config\microscope-configuration-demo2.yaml"
)

# load new style protocol with adaptive-polish with validation
PROTOCOL_PATH = "protocol-on-grid-ap-only.yaml"
protocol = validate_protocol(utils.load_protocol(protocol_path=PROTOCOL_PATH))

2025-03-14 15:20:06,104 — root — INFO — connect_to_microscope:5418 — Microscope client connected to DemoMicroscope with serial number 123456 and software version 0.1
2025-03-14 15:20:06,104 — root — INFO — setup_session:229 — Finished setup for session: demo_2025-03-14-03-20-06PM
2025-03-14 15:20:06,114 — root — INFO — validate_protocol:166 — Adding default milling stage for mill_rough
2025-03-14 15:20:06,118 — root — INFO — validate_protocol:166 — Adding default milling stage for microexpansion
2025-03-14 15:20:06,118 — root — INFO — validate_protocol:166 — Adding default milling stage for fiducial
2025-03-14 15:20:06,118 — root — INFO — validate_protocol:166 — Adding default milling stage for notch


In [ ]:
# acquire reference images (required to register paths)
lamella_folder = r"C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\tmp\AutoLamella-2025-03-10-22-34\01-strong-salmon"
settings.image.path = lamella_folder
acquire.take_reference_images(microscope, settings.image)

# set imaging settings folder on fib so that adaptive polish finds the correct lamella folder
fib_img_settings = microscope.get_imaging_settings(BeamType.ION)
fib_img_settings.path = lamella_folder
microscope._last_imaging_settings = fib_img_settings

milling_stages = get_milling_stages("mill_polishing", protocol["milling"])
strategy_config: strategy.AdaptivePolishMillingConfig = milling_stages[0].strategy.config
print("Adaptive Polishing Config:")
print(f"Milling Interval: {strategy_config.milling_interval_s}")
print(f"Maximum Cycles: {strategy_config.max_milling_cycles}")
print(f"Model: {strategy_config.model_path}")

In [ ]:
# run milling stages
mill_stages(microscope=microscope, stages=milling_stages)

In [ ]:
# for more detailed error finding
# milling_stages[0].strategy.run(microscope, milling_stages[0])

## Setting up with autolamella

In [ ]:
from autolamella.structures import Experiment, AutoLamellaProtocol
from pathlib import Path

In [12]:
EXP_PATH = "../../../../tmp/AutoLamella-2025-03-10-22-34/experiment.yaml"
PROTOCOL_PATH = "protocol-on-grid-ap.yaml"
exp = Experiment.load(EXP_PATH)
pos = exp.positions[0]
protocol = AutoLamellaProtocol.load(PROTOCOL_PATH)
print(protocol.method.workflow)

print(f"Last Completed: {pos.last_completed}")
print(f"Next: ", protocol.method.get_next(pos.workflow))
print(f"Previous: ", protocol.method.get_previous(pos.workflow))
print(f"Workflow: ", protocol.method.workflow)

2025-03-14 16:28:11,830 — root — INFO — validate_protocol:166 — Adding default milling stage for notch
[<AutoLamellaStage.SetupLamella: 7>, <AutoLamellaStage.MillRough: 8>, <AutoLamellaStage.SetupPolishing: 9>, <AutoLamellaStage.MillPolishing: 10>]
Last Completed: Created (in progress)
Next:  SetupLamella
Previous:  Created
Workflow:  [<AutoLamellaStage.SetupLamella: 7>, <AutoLamellaStage.MillRough: 8>, <AutoLamellaStage.SetupPolishing: 9>, <AutoLamellaStage.MillPolishing: 10>]


In [20]:
lamella_folder = Path(f"{exp.path}/{exp.positions[0].petname}")
settings.image.path = lamella_folder
acquire.take_reference_images(microscope, settings.image)

# set imaging settings folder on fib so that adaptive polish finds the correct lamella folder
fib_img_settings = microscope.get_imaging_settings(BeamType.ION)
fib_img_settings.path = lamella_folder
microscope._last_imaging_settings = fib_img_settings

milling_stages = protocol.milling["mill_polishing"]
strategy_config: strategy.AdaptivePolishMillingConfig = milling_stages[0].strategy.config
print("Adaptive Polishing Config:")
print(f"Milling Interval: {strategy_config.milling_interval_s}")
print(f"Maximum Cycles: {strategy_config.max_milling_cycles}")
print(f"Model: {strategy_config.model_path}")

2025-03-14 16:40:36,024 — root — INFO — acquire_image:6335 — acquiring new ELECTRON image.
2025-03-14 16:40:36,024 — root — INFO — acquire_image:6336 — resolution:[1536, 1024], hfw:0.00015
2025-03-14 16:40:36,024 — root — INFO — acquire_image:6343 — SEM
2025-03-14 16:40:36,024 — root — INFO — load:141 — Loading C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2024\00_Adaptive-milling\test_images\test_07-steady-gator\SEM\adapt_mill_img_001_electron_001.tif as spoof image
2025-03-14 16:40:36,125 — root — INFO — acquire_image:6370 — img_data.type:<class 'numpy.ndarray'>
2025-03-14 16:40:36,127 — root — INFO — acquire_image:6371 — img_data.dtype:uint8, img_data.shape:(2048, 3072)
2025-03-14 16:40:36,128 — root — INFO — acquire_image:6375 — demo image has shape:(2048, 3072). Resizing to (np.int64(1024), np.int64(1536))
2025-03-14 16:40:36,341 — root — INFO — acquire_image:6379 — Resized, img_data.dtype:uint8, img_data.shape:(1024, 1536)
2025-03-14 16:40:36,341 — root — INFO — ac

In [21]:
# run milling stages
mill_stages(microscope=microscope, stages=milling_stages)

2025-03-14 16:41:04,812 — root — INFO — run:100 — Running Adaptive polishing according to GIS thickness for Polishing Mill 01
2025-03-14 16:41:04,812 — root — WARNING — _set:6184 — Unknown key: active_device (BeamType.ION)
2025-03-14 16:41:04,812 — root — INFO — run:111 — Lamella folder ..\..\..\..\tmp\AutoLamella-2025-03-10-22-34\01-strong-salmon/adaptive_polish already exists, some data may be overwritten.
2025-03-14 16:41:04,822 — root — INFO — __init__:174 — model path:C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\models\2024-02-24_0013_gis_lamela_crack_pytorch_AUnet.ptchkp
2025-03-14 16:41:05,922 — root — INFO — __init__:180 — DL device: cuda
2025-03-14 16:41:06,661 — root — INFO — run:124 — Using sem beam shift alignment for adaptive polishing
2025-03-14 16:41:06,661 — root — INFO — acquire_image:6335 — acquiring new ELECTRON image.
2025-03-14 16:41:06,661 — root — INFO — acquire_image:6336 — resolution:[1536, 1024], hfw:0.00015
2025-03-14 

C:\Users\dmv31621\OneDrive - The Rosalind Franklin Institute\2025\00_Adaptive-milling\adaptive_polish\src\adaptive_polish\gis_measurement.py:147: RuntimeWarning: All-NaN slice encountered
  GIS_windowed = np.nanmedian(GIS_pxbypx_NaNed.reshape(-1, window_size_px), axis=1)


2025-03-14 16:41:10,509 — root — INFO — run:263 — Stopping as crack area (um2) 2.0980834960937496 > threshold 2 um2
2025-03-14 16:41:10,581 — root — INFO — finish_milling:86 — Changing to Imaging Current: 2.00e-11
2025-03-14 16:41:10,581 — root — INFO — finish_milling:5724 — Finishing milling: 2.00e-11
2025-03-14 16:41:10,581 — root — INFO — finish_milling:88 — Finished Ion Beam Milling.
2025-03-14 16:41:10,581 — root — INFO — finish_milling:86 — Changing to Imaging Current: 2.00e-11
2025-03-14 16:41:10,581 — root — INFO — finish_milling:5724 — Finishing milling: 2.00e-11
2025-03-14 16:41:10,581 — root — INFO — finish_milling:88 — Finished Ion Beam Milling.
